In [3]:
#Bài tập ở lớp
import copy
from heapq import heappush, heappop

# Đổi n = 4 cho bài toán 15-puzzle
n = 4

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

def calculateCosts(mats, final) -> int:
    """
    Sử dụng khoảng cách Manhattan thay vì đếm số ô sai vị trí
    để thuật toán AKT chạy nhanh hơn và hiệu quả hơn với n=4.
    """
    dist = 0
    # Tạo một bản đồ vị trí các số trong ma trận đích để tra cứu nhanh
    final_pos = {}
    for i in range(n):
        for j in range(n):
            final_pos[final[i][j]] = (i, j)

    for i in range(n):
        for j in range(n):
            val = mats[i][j]
            if val != 0:  # Không tính ô trống
                target_x, target_y = final_pos[val]
                dist += abs(i - target_x) + abs(j - target_y)
    return dist

def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n

class Nodes:
    def __init__(self, parent, mats, empty_tile_posi, h_cost, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.h_cost = h_cost  # h(n)
        self.levels = levels  # g(n)
        self.f_cost = h_cost + levels  # f(n) = g(n) + h(n)

    # So sánh dựa trên f(n) để ưu tiên các nút có tổng chi phí thấp nhất
    def __lt__(self, nxt):
        return self.f_cost < nxt.f_cost

def printMatrix(mats):
    for row in mats:
        print(" ".join(f"{val:2d}" for val in row))
    print()

def printPath(root):
    if root is None:
        return
    printPath(root.parent)
    printMatrix(root.mats)

def solve(initial, empty_tile_posi, final):
    pq = [] # Sử dụng list trực tiếp cho heapq

    h = calculateCosts(initial, final)
    root = Nodes(None, initial, empty_tile_posi, h, 0)
    heappush(pq, root)

    # Tập hợp các trạng thái đã đi qua để tránh lặp vô hạn (Quan trọng cho n=4)
    visited = set()

    while pq:
        minimum = heappop(pq)

        # Chuyển ma trận thành tuple để lưu vào set
        mats_tuple = tuple(tuple(row) for row in minimum.mats)
        if mats_tuple in visited:
            continue
        visited.add(mats_tuple)

        # Nếu h(n) == 0 tức là đã đạt tới đích
        if minimum.h_cost == 0:
            printPath(minimum)
            return

        for i in range(4):
            new_x = minimum.empty_tile_posi[0] + rows[i]
            new_y = minimum.empty_tile_posi[1] + cols[i]

            if isSafe(new_x, new_y):
                # Tạo ma trận mới
                new_mats = copy.deepcopy(minimum.mats)
                x1, y1 = minimum.empty_tile_posi
                new_mats[x1][y1], new_mats[new_x][new_y] = new_mats[new_x][new_y], new_mats[x1][y1]

                h_child = calculateCosts(new_mats, final)
                child = Nodes(minimum, new_mats, [new_x, new_y], h_child, minimum.levels + 1)
                heappush(pq, child)

# --- Chạy thử nghiệm ---
# 0 đại diện cho ô trống
initial = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 0, 11],
    [13, 14, 15, 12]
]

final = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

# Tìm vị trí số 0 trong initial
empty_pos = [2, 2]

solve(initial, empty_pos, final)

 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12

 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12

 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0

